# ATProto Visualizers

This notebook demonstrates the JupyterLite ATProto visualizer extensions:
- **CID Decoder** - structured display of CID components
- **File Upload** - upload .car/.cbor/.cid files to the browser filesystem

**What you'll learn:**
- How CID strings are decoded and displayed
- How to upload ATProto data files
- How the kernel CID output triggers the viewer

**Estimated Time:** 5-10 minutes

## Step 1: CID Strings in the Kernel

When the kernel outputs a CID string, the CID viewer automatically renders it.
CIDv1 strings start with 'b' (base32 encoding) and CIDv0 strings start with 'Qm' (base58btc).

Try running the cell below - the output should render as a structured CID table.

In [ ]:
// CIDv1 base32 - the most common format in ATProto
// bafyrei = CIDv1, dag-pb codec, sha2-256 hash
NSString *cid = @"bafyreib4ci6gyh6jhdz6q4kf3fmczq2i4e2n5j7k3g6t5h4m2n1j3k5l7";
cid

## Step 2: CIDv0 (Legacy IPFS Format)

CIDv0 uses base58btc encoding and always starts with 'Qm'. These are less common in ATProto but still supported.

In [ ]:
// CIDv0 base58btc - legacy IPFS multihash
NSString *cidv0 = @"QmYwAPJzv5CZsnA625s3Xf2nemtYgPpHdWEz79ojWnPbdg";
cidv0

## Step 3: Real ATProto CIDs

These are CIDs you would encounter in actual ATProto repository data. The viewer decodes the codec, hash algorithm, and digest.

In [ ]:
// A commit CID from a real PDS repository
@"bafyreihr4eomkqj5f3g4c2n7x6k8l5m3p9q1r2s4t6u8v0w2y4z6a8b0c2d4"

In [ ]:
// A record CID (dag-cbor codec)
@"bafy2bzacectr2fwrxgb3s7n3n5v4c2y6x7z8a9b0c1d2e3f4g5h6i7j8k9l0"

## Step 4: CID Helper Class

Build a simple CID class that produces base32-encoded CIDv1 strings from SHA-256 digests. The host bridge provides the SHA-256 computation.

In [ ]:
@interface CIDBuilder : NSObject
- (NSString *)cidForData:(NSData *)data;
- (NSString *)cidForString:(NSString *)str;
@end

@implementation CIDBuilder
- (NSString *)cidForData:(NSData *)data {
    NSData *digest = [CID sha256Digest:data];
    // Build a CIDv1 base32 string: bafkrei + hex digest
    // (Simplified - real CID uses varint encoding + multihash)
    NSString *hex = [digest description];
    return [NSString stringWithFormat:@"bafkrei%@", hex];
}
- (NSString *)cidForString:(NSString *)str {
    NSData *data = [NSData dataWithBytes:[str UTF8String] length:[str length]];
    return [self cidForData:data];
}
@end

CIDBuilder *builder = [[CIDBuilder alloc] init];
NSString *profileCid = [builder cidForString:@"hello atproto"];
NSLog(@"Profile CID: %@", profileCid);

## Step 5: File Upload

Use the **Upload ATProto File** command in the command palette (Ctrl+Shift+C) to upload .car, .cbor, or .cid files to the browser filesystem.

Once uploaded, you can reference these files from kernel code. The file browser in the left sidebar will show uploaded files with their correct type icons.

In [ ]:
// Verify the upload plugin registered the file types
// by checking that .car files appear in the file browser
NSLog(@"Upload .car files via: Command Palette > Upload ATProto File");
NSLog(@"Or drag and drop into the file browser");
NSLog(@"Supported: .car, .cbor, .cid");

## Step 6: CID Anatomy

A CIDv1 has three binary components:
1. **Version** (varint): 1
2. **Codec** (varint): 0x71 = dag-cbor, 0x55 = raw
3. **Multihash**: code 0x12 = sha2-256, then digest

The viewer decodes all of these from the base32 string.

In [ ]:
// Demonstrate different codec types
// dag-cbor (0x71) - the standard ATProto record codec
@"bafyreib4ci6gyh6jhdz6q4kf3fmczq2i4e2n5j7k3g6t5h4m2n1j3k5l7"